In [1]:
# ==========================================
# PREPARACIÓN DEL ENTORNO
# ==========================================
using Pkg
Pkg.activate("/home/antonibancells/Desktop/Codi/TFM/ProvesModernes")

using ITensors
using ITensorMPS
using Plots
using Printf

  Activating project at `~/Desktop/Codi/TFM/ProvesModernes`


El estado cuántico $|\psi\rangle$ se implementa mediante un objeto ITensor. Sus índices se definen mediante siteinds de tipo "Qubit". Para introducir valores en el tensor, es necesario realizar una codificación binaria, que puede hacerse de dos maneras:
<ul>
    <li>MSB en la posición 1</li>
    <li>LSB en la posición 1</li>
</ul>
En el presente ejemplo se realiza la codificación con LSB en la posición 1.

Para el cálculo de la derivada se necesita la implementación de los operadors de desplazamiento, ya sea $F|i\rangle=|i+1\rangle$ (FSB( o $B|i\rangle=|i-1\rangle$ (BSB). En el presente trabajo se implementa la segunda opción.

En este caso, el operador derivada queda expresado como $$D^{(1)}=\frac{E-2I+E^\dagger}{2dx}$$

In [3]:
function construir_derivada2_mpo(sites, dx)
    N = length(sites)
    tensors_E = Vector{ITensor}(undef, N)
    
    # links_E[j] conectará el sitio j con el j+1
    links_E = [Index(2, "LinkE,l=$i") for i in 1:(N-1)]
    
    # El sitio 1 es el LSB (aquí empieza la resta)
    # El acarreo (deuda) viaja de izquierda a derecha (de 1 a N)
    for j in 1:N
        s = sites[j]
        sp = s'
        
        # ---------------------------------------------------------------------
        # CASO A: LSB (Sitio 1) - Aquí se inicia la resta de -1
        # ---------------------------------------------------------------------
        if j == 1  
            # Conecta con el sitio 2 a través de links_E[1]
            T = ITensor(s, sp, links_E[j])
            
            # |1> (físico 2) - 1 = |0> (físico 1). No hay deuda.
            T[s=>2, sp=>1, links_E[j]=>1] = 1.0  
            # |0> (físico 1) - 1 = |1> (físico 2). Se genera deuda (estado 2).
            T[s=>1, sp=>2, links_E[j]=>2] = 1.0  
            
            tensors_E[j] = T
            
        # ---------------------------------------------------------------------
        # CASO B: MSB (Sitio N) - Final de la propagación de deuda
        # ---------------------------------------------------------------------
        elseif j == N  
            # Viene del sitio N-1 a través de links_E[N-1]
            T = ITensor(links_E[j-1], s, sp)
            
            # Si NO viene deuda (link => 1), se queda igual
            T[links_E[j-1]=>1, s=>1, sp=>1] = 1.0
            T[links_E[j-1]=>1, s=>2, sp=>2] = 1.0
            
            # Si SÍ viene deuda (link => 2), restamos 1
            T[links_E[j-1]=>2, s=>2, sp=>1] = 1.0  # |1> -> |0>
            T[links_E[j-1]=>2, s=>1, sp=>2] = 1.0  # |0> -> |1> (Periódico)
            
            tensors_E[j] = T
            
        # ---------------------------------------------------------------------
        # CASO C: Sitios intermedios
        # ---------------------------------------------------------------------
        else  
            # links_E[j-1] viene de la izquierda, links_E[j] va hacia la derecha
            T = ITensor(links_E[j-1], s, sp, links_E[j])
            
            # Si NO viene deuda
            T[links_E[j-1]=>1, s=>1, sp=>1, links_E[j]=>1] = 1.0
            T[links_E[j-1]=>1, s=>2, sp=>2, links_E[j]=>1] = 1.0
            
            # Si SÍ viene deuda
            T[links_E[j-1]=>2, s=>2, sp=>1, links_E[j]=>1] = 1.0 # Deuda resuelta
            T[links_E[j-1]=>2, s=>1, sp=>2, links_E[j]=>2] = 1.0 # Deuda se propaga
            
            tensors_E[j] = T
        end
    end
    
    E = MPO(tensors_E)
    E_dag = swapprime(conj(E), 0 => 1)
    I_mpo=MPO(sites,"Id")
    
    # Operador derivada segunda
    D2_mpo = add((1.0 / (dx^2)) * E, (1.0 / (dx^2)) * E_dag,(-2.0 / (dx^2)) * I_mpo; cutoff=1e-12)
    
    return D2_mpo
end

function main()
    N = 10                  
    dim_total = 1 << N      
    x_min = 0.0
    x_max = 2 * π           
    dx = (x_max - x_min) / dim_total 

    sites = siteinds("Qubit", N)

    psi_tensor = ITensor(sites...)
    
    # Rellenamos el tensor mapeando: Sitio 1 = LSB, Sitio N = MSB
    for i in 0:(dim_total - 1)
        x = x_min + i * dx
        val = sin(x)
        
        # j=1 es el LSB (i >> 0), j=N es el MSB (i >> N-1)
        estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
        psi_tensor[estado_binario...] = val
    end
    
    psi = MPS(psi_tensor, sites; cutoff=1e-12, maxdim=64)

    # Llamamos a nuestro MPO 
    D2_mpo = construir_derivada2_mpo(sites, dx)
    println("-> MPO de la derivada segunda generado con éxito. Rango: ", maxlinkdim(D2_mpo))

    # Aplicamos el MPO
    d2_psi = contract(D2_mpo, psi; cutoff=1e-12)
    d2_psi = noprime(d2_psi)

    # Reconstruimos el tensor para verificar
    res_tensor = d2_psi[1]
    for j in 2:N 
        res_tensor *= d2_psi[j] 
    end

    println("\nComprobación de los resultados:")
        
    for i in 0:100:1000
        x = x_min + i * dx
        # Usamos exactamente el mismo mapeo de bits para leer el resultado
        estado_binario = [((i >> (j - 1)) & 1) + 1 for j in 1:N]
        val_qtt = res_tensor[estado_binario...]
        @printf("x = %7.4f | QTT: %7.4f | Real : %7.4f | Error: %7.4e\n", 
                x, val_qtt, -sin(x), abs(val_qtt +sin(x)))
    end
end

main()

-> MPO de la derivada segunda generado con éxito. Rango: 3

Comprobación de los resultados:
x =  0.0000 | QTT:  0.0000 | Real : -0.0000 | Error: 1.7270e-09
x =  0.6136 | QTT: -0.5758 | Real : -0.5758 | Error: 1.8045e-06
x =  1.2272 | QTT: -0.9415 | Real : -0.9415 | Error: 2.9560e-06
x =  1.8408 | QTT: -0.9638 | Real : -0.9638 | Error: 3.0267e-06
x =  2.4544 | QTT: -0.6344 | Real : -0.6344 | Error: 1.9873e-06
x =  3.0680 | QTT: -0.0736 | Real : -0.0736 | Error: 2.3090e-07
x =  3.6816 | QTT:  0.5141 | Real :  0.5141 | Error: 1.6129e-06
x =  4.2951 | QTT:  0.9142 | Real :  0.9142 | Error: 2.8713e-06
x =  4.9087 | QTT:  0.9808 | Real :  0.9808 | Error: 3.0743e-06
x =  5.5223 | QTT:  0.6895 | Real :  0.6895 | Error: 2.1615e-06
x =  6.1359 | QTT:  0.1467 | Real :  0.1467 | Error: 4.6247e-07
